# Programación GPU con Python — CUDA desde Google Colab

> **Curso: Fundamentos de Cómputo de Alto Desempeño** | Material práctico complementario

Este notebook implementa los conceptos teóricos de CUDA usando Python con:
- **CuPy**: NumPy acelerado por GPU
- **Numba CUDA**: kernels CUDA escritos en Python puro
- **PyTorch**: operaciones tensoriales GPU

---

### 📋 Contenidos
1. Configuración del entorno y verificación de GPU
2. **1D** — Suma de vectores y operaciones elementales
3. **1D** — Reducción paralela con memoria compartida
4. **2D** — Procesamiento de imágenes (grayscale, blur, inversión)
5. **2D** — Multiplicación de matrices (benchmark)
6. **3D** — Operaciones sobre tensores (batch norm, ReLU, CNN feature maps)
7. Benchmarking sistemático CPU vs GPU
8. GPU en machine learning con PyTorch
9. Kernel CUDA C embebido en Python (CuPy RawKernel)

---

NOTA: **Requisito**: En Colab ir a `Entorno de ejecución → Cambiar tipo → T4 GPU` antes de ejecutar.

---
## Sección 0 — Configuración del entorno

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('❌ No se detectó GPU. Ir a Entorno de ejecución → Cambiar tipo → T4 GPU')

In [ ]:
!pip install cupy-cuda12x -q
print('✅ CuPy instalado')

In [ ]:
import numpy as np
import cupy as cp
import time
import math
import matplotlib.pyplot as plt
from numba import cuda
import torch

device = cp.cuda.Device(0)
attrs  = device.attributes
props  = cp.cuda.runtime.getDeviceProperties(0)

print('=' * 55)
print('           INFORMACIÓN DE LA GPU')
print('=' * 55)
print(f'  Nombre       : {props["name"].decode()}')
print(f'  VRAM total   : {device.mem_info[1] / 1024**3:.1f} GB')
print(f'  VRAM libre   : {device.mem_info[0] / 1024**3:.2f} GB')
print(f'  CUDA Compute : {attrs["ComputeCapabilityMajor"]}.{attrs["ComputeCapabilityMinor"]}')
print(f'  Max hilos/bloque  : {attrs["MaxThreadsPerBlock"]}')
print(f'  Max hilos/SM      : {attrs["MaxThreadsPerMultiProcessor"]}')
print(f'  Multiprocesadores : {attrs["MultiProcessorCount"]}')
print(f'  Mem compartida/SM : {attrs["MaxSharedMemoryPerBlock"] / 1024:.0f} KB')
print('=' * 55)
print(f'  PyTorch CUDA  : {torch.cuda.is_available()}')
print(f'  Torch device  : {torch.cuda.get_device_name(0)}')

---
## Sección 1 — Dimensión 1D: Operaciones sobre Vectores

Los datos 1D son el caso más simple: arrays, señales, series temporales.
El índice global de cada hilo es:

```
i = blockIdx.x × blockDim.x + threadIdx.x
```

In [ ]:
# ── EJEMPLO 1A: Suma de vectores con CuPy ──────────────────────────
# CuPy gestiona automáticamente H2D, kernel y D2H

N = 10_000_000  # 10 millones de elementos

h_A = np.random.randn(N).astype(np.float32)
h_B = np.random.randn(N).astype(np.float32)

# CPU secuencial
t0 = time.perf_counter()
h_C_cpu = h_A + h_B
t_cpu = time.perf_counter() - t0

# GPU con CuPy
# Paso 1: H2D — transferir a VRAM
d_A = cp.asarray(h_A)
d_B = cp.asarray(h_B)

# Warmup (primera llamada tiene overhead de compilación JIT)
_ = d_A + d_B
cp.cuda.Stream.null.synchronize()

# Medición real
t0 = time.perf_counter()
d_C = d_A + d_B                          # kernel interno de CuPy
cp.cuda.Stream.null.synchronize()        # equivale a cudaDeviceSynchronize
t_gpu = time.perf_counter() - t0

# Paso D2H — recuperar resultados
h_C_gpu = cp.asnumpy(d_C)

print(f'Suma de vectores — N = {N:,}')
print(f'  Resultado correcto : {np.allclose(h_C_cpu, h_C_gpu)}')
print(f'  Tiempo CPU         : {t_cpu*1000:.2f} ms')
print(f'  Tiempo GPU (kernel): {t_gpu*1000:.2f} ms')
print(f'  Speedup            : {t_cpu/t_gpu:.1f}x')

In [ ]:
# ── EJEMPLO 1B: Kernel CUDA en Python con Numba ──────────────────────
# Aquí escribimos el kernel manualmente, igual que en C/CUDA

from numba import cuda

@cuda.jit
def kernel_add_vectors(A, B, C, N):
    """
    Kernel 1D: suma de vectores
    C equivale a:  __global__ void addVectors(float* A, float* B, float* C, int N)
    """
    # Calcular índice global (idéntico a C/CUDA)
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i < N:
        C[i] = A[i] + B[i]


@cuda.jit
def kernel_scale_and_shift(data, out, scale, shift, N):
    """Kernel 1D: transformación lineal  out[i] = scale * data[i] + shift"""
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i < N:
        out[i] = scale * data[i] + shift


N = 1_000_000
h_A = np.random.randn(N).astype(np.float32)
h_B = np.random.randn(N).astype(np.float32)

# H2D
d_A = cuda.to_device(h_A)
d_B = cuda.to_device(h_B)
d_C = cuda.device_array(N, dtype=np.float32)

# Configuración de lanzamiento
BLOCK_SIZE = 256
GRID_SIZE  = math.ceil(N / BLOCK_SIZE)
print(f'Lanzamiento: grid={GRID_SIZE} × {BLOCK_SIZE} hilos = {GRID_SIZE*BLOCK_SIZE:,} hilos totales')

# Lanzar kernel
kernel_add_vectors[GRID_SIZE, BLOCK_SIZE](d_A, d_B, d_C, N)
cuda.synchronize()

h_C = d_C.copy_to_host()
print(f'Resultado correcto: {np.allclose(h_C, h_A + h_B)}')

# Kernel de transformación lineal: mapea [0,1] → [-1,1]
data_in  = cuda.to_device(np.linspace(0, 1, N, dtype=np.float32))
data_out = cuda.device_array(N, dtype=np.float32)
kernel_scale_and_shift[GRID_SIZE, BLOCK_SIZE](data_in, data_out, 2.0, -1.0, N)
cuda.synchronize()
result = data_out.copy_to_host()
print(f'Transformación lineal — primeros 5: {result[:5]}')
print(f'  (esperado: de 0..1 mapeado a -1..1)')

In [ ]:
# ── EJEMPLO 1C: Reducción paralela (suma global) ───────────────────
# Patrón más complejo: los hilos cooperan con memoria compartida

@cuda.jit
def kernel_sum_reduction(data, partial_sums, N):
    """
    Reducción paralela usando __shared__.
    Cada bloque produce una suma parcial.
    """
    sdata = cuda.shared.array(shape=256, dtype=np.float32)

    tid = cuda.threadIdx.x
    i   = cuda.blockIdx.x * cuda.blockDim.x + tid

    # Cargar dato en shared memory
    sdata[tid] = data[i] if i < N else 0.0
    cuda.syncthreads()

    # Reducción por mitades: stride = 128, 64, 32, ..., 1
    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tid < stride:
            sdata[tid] += sdata[tid + stride]
        cuda.syncthreads()
        stride //= 2

    # Hilo 0 escribe la suma del bloque
    if tid == 0:
        partial_sums[cuda.blockIdx.x] = sdata[0]


N = 1_048_576  # 2^20
h_data = np.ones(N, dtype=np.float32) * 2.0  # todos = 2 → suma esperada = 2*N

d_data    = cuda.to_device(h_data)
BLOCK     = 256
GRID      = N // BLOCK
d_partial = cuda.device_array(GRID, dtype=np.float32)

kernel_sum_reduction[GRID, BLOCK](d_data, d_partial, N)
cuda.synchronize()

# Sumar parciales en CPU
total = float(np.sum(d_partial.copy_to_host()))

print(f'Reducción paralela — N = {N:,}')
print(f'  Suma GPU   = {total:.0f}')
print(f'  Suma CPU   = {np.sum(h_data):.0f}')
print(f'  Correcto   = {np.isclose(total, np.sum(h_data))}')
print(f'  Bloques de reducción: {GRID}')

---
## Sección 2 — Dimensión 2D: Procesamiento de Imágenes y Matrices

En 2D, cada hilo se identifica por `(row, col)`:
```
col = blockIdx.x * blockDim.x + threadIdx.x
row = blockIdx.y * blockDim.y + threadIdx.y
idx = row * width + col          # índice lineal en memoria
```

In [ ]:
# ── EJEMPLO 2A: Kernel 2D — grayscale e inversión ──────────────────

@cuda.jit
def kernel_grayscale(img_in, img_out, H, W):
    """
    Kernel 2D: cada hilo procesa un pixel (row, col).
    Fórmula BT.601: Y = 0.299R + 0.587G + 0.114B
    """
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    if col < W and row < H:
        r = img_in[row, col, 0]
        g = img_in[row, col, 1]
        b = img_in[row, col, 2]
        img_out[row, col] = np.uint8(0.299 * r + 0.587 * g + 0.114 * b)


@cuda.jit
def kernel_invert(img_in, img_out, H, W):
    """Kernel 2D: inversión de colores (negativo fotográfico)"""
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    if col < W and row < H:
        img_out[row, col, 0] = 255 - img_in[row, col, 0]
        img_out[row, col, 1] = 255 - img_in[row, col, 1]
        img_out[row, col, 2] = 255 - img_in[row, col, 2]


def make_test_image(H, W):
    img = np.zeros((H, W, 3), dtype=np.uint8)
    img[:, :, 0] = np.linspace(0, 255, W, dtype=np.uint8)
    img[:, :, 1] = np.linspace(0, 255, H, dtype=np.uint8)[:, None]
    img[:, :, 2] = 128
    return img


H, W = 2048, 2048
h_img = make_test_image(H, W)

BLOCK_2D = (16, 16)
GRID_2D  = (math.ceil(W / 16), math.ceil(H / 16))
print(f'Imagen: {W}x{H} = {W*H:,} pixeles')
print(f'Bloques 2D: {GRID_2D[0]}x{GRID_2D[1]} = {GRID_2D[0]*GRID_2D[1]:,} bloques')
print(f'Hilos totales: {GRID_2D[0]*GRID_2D[1]*16*16:,}')

d_img      = cuda.to_device(h_img)
d_gray     = cuda.device_array((H, W), dtype=np.uint8)
d_inverted = cuda.device_array_like(h_img)

t0 = time.perf_counter()
kernel_grayscale[GRID_2D, BLOCK_2D](d_img, d_gray, H, W)
cuda.synchronize()
t_gray = time.perf_counter() - t0

kernel_invert[GRID_2D, BLOCK_2D](d_img, d_inverted, H, W)
cuda.synchronize()

h_gray     = d_gray.copy_to_host()
h_inverted = d_inverted.copy_to_host()
print(f'Tiempo kernel grayscale ({W}x{H}): {t_gray*1000:.2f} ms')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(h_img);               axes[0].set_title('Original (RGB)');       axes[0].axis('off')
axes[1].imshow(h_gray, cmap='gray'); axes[1].set_title('Escala de grises GPU'); axes[1].axis('off')
axes[2].imshow(h_inverted);          axes[2].set_title('Invertida GPU');         axes[2].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── EJEMPLO 2B: Filtro de convolución 2D (box blur) ────────────────

@cuda.jit
def kernel_box_blur(img_in, img_out, H, W, radius):
    """
    Kernel 2D: box blur (promedio de vecindad).
    Cada hilo promedia los pixeles dentro del radio dado.
    """
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    if col < W and row < H:
        r_sum = g_sum = b_sum = 0.0
        count = 0
        for dy in range(-radius, radius + 1):
            for dx in range(-radius, radius + 1):
                ny, nx = row + dy, col + dx
                if 0 <= ny < H and 0 <= nx < W:
                    r_sum += img_in[ny, nx, 0]
                    g_sum += img_in[ny, nx, 1]
                    b_sum += img_in[ny, nx, 2]
                    count += 1
        if count > 0:
            img_out[row, col, 0] = np.uint8(r_sum / count)
            img_out[row, col, 1] = np.uint8(g_sum / count)
            img_out[row, col, 2] = np.uint8(b_sum / count)


H2, W2 = 512, 512
h_img2    = make_test_image(H2, W2)
d_img2    = cuda.to_device(h_img2)
d_blurred = cuda.device_array_like(h_img2)
GRID2     = (math.ceil(W2 / 16), math.ceil(H2 / 16))

# Medir tiempos con distintos radios
radii_times = {}
for r in [1, 3, 5, 10]:
    t0 = time.perf_counter()
    kernel_box_blur[GRID2, (16, 16)](d_img2, d_blurred, H2, W2, r)
    cuda.synchronize()
    radii_times[r] = (time.perf_counter() - t0) * 1000

print('Tiempo de blur por radio (512x512):')
for r, t in radii_times.items():
    print(f'  radius={r:2d}  ->  {t:.2f} ms')

# Visualizar con radio=5
kernel_box_blur[GRID2, (16, 16)](d_img2, d_blurred, H2, W2, 5)
cuda.synchronize()
h_blurred = d_blurred.copy_to_host()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(h_img2);    ax1.set_title('Original');          ax1.axis('off')
ax2.imshow(h_blurred); ax2.set_title('Box Blur GPU (r=5)'); ax2.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── EJEMPLO 2C: Multiplicación de matrices — benchmark CPU vs GPU ────

def benchmark_matmul(size):
    A_cpu = np.random.randn(size, size).astype(np.float32)
    B_cpu = np.random.randn(size, size).astype(np.float32)

    t0 = time.perf_counter()
    _ = np.dot(A_cpu, B_cpu)
    t_cpu = (time.perf_counter() - t0) * 1000

    A_gpu = cp.asarray(A_cpu)
    B_gpu = cp.asarray(B_cpu)
    _ = cp.dot(A_gpu, B_gpu); cp.cuda.Stream.null.synchronize()  # warmup

    t0 = time.perf_counter()
    _ = cp.dot(A_gpu, B_gpu)
    cp.cuda.Stream.null.synchronize()
    t_gpu = (time.perf_counter() - t0) * 1000

    return t_cpu, t_gpu


sizes   = [128, 256, 512, 1024, 2048, 4096]
results = {s: benchmark_matmul(s) for s in sizes}

print(f'{"Tamaño":>8}  {"CPU (ms)":>10}  {"GPU (ms)":>10}  {"Speedup":>8}')
print('-' * 45)
for s, (tc, tg) in results.items():
    print(f'{s:>8}  {tc:>10.2f}  {tg:>10.2f}  {tc/tg:>7.1f}x')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(sizes, [results[s][0] for s in sizes], 'o-', color='#F59E0B', label='CPU', lw=2)
ax1.plot(sizes, [results[s][1] for s in sizes], 's-', color='#00D4FF', label='GPU (cuBLAS)', lw=2)
ax1.set_xlabel('N'); ax1.set_ylabel('ms'); ax1.set_title('Tiempo: Matmul N×N')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.bar(sizes, [results[s][0]/results[s][1] for s in sizes], color='#10B981', alpha=0.8)
ax2.set_xlabel('N'); ax2.set_ylabel('Speedup (x)'); ax2.set_title('Speedup GPU vs CPU')
ax2.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## Sección 3 — Dimensión 3D: Tensores y Feature Maps

En 3D, cada hilo procesa un elemento `(z, y, x)`:
```
x   = blockIdx.x * blockDim.x + threadIdx.x
y   = blockIdx.y * blockDim.y + threadIdx.y
z   = blockIdx.z * blockDim.z + threadIdx.z
idx = z * (dimX * dimY) + y * dimX + x
```
Aplicaciones: CNNs (batch × canales × H × W), simulaciones volumétricas, física 3D.

In [ ]:
# ── EJEMPLO 3A: Kernels 3D — batch normalize y ReLU ────────────────

@cuda.jit
def kernel_batch_normalize_3d(tensor, mean, std, out, B, H, W):
    """
    Kernel 3D: normalización sobre tensor (B, H, W).
    Cada hilo procesa un elemento (b, row, col).
    """
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x  # dim W
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y  # dim H
    b   = cuda.blockIdx.z * cuda.blockDim.z + cuda.threadIdx.z  # dim Batch

    if col < W and row < H and b < B:
        out[b, row, col] = (tensor[b, row, col] - mean) / (std + 1e-8)


@cuda.jit
def kernel_relu_3d(tensor, out, B, H, W):
    """Kernel 3D: activación ReLU — out[b,h,w] = max(0, tensor[b,h,w])"""
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    b   = cuda.blockIdx.z * cuda.blockDim.z + cuda.threadIdx.z

    if col < W and row < H and b < B:
        val = tensor[b, row, col]
        out[b, row, col] = val if val > 0.0 else 0.0


BATCH, IMG_H, IMG_W = 32, 128, 128
h_tensor = np.random.randn(BATCH, IMG_H, IMG_W).astype(np.float32)
print(f'Tensor shape: {h_tensor.shape}  ->  {h_tensor.size:,} elementos')

d_tensor   = cuda.to_device(h_tensor)
d_out_bn   = cuda.device_array_like(h_tensor)
d_out_relu = cuda.device_array_like(h_tensor)

# Configuración 3D: blockDim (16, 16, 1) = 256 hilos/bloque
BLOCK_3D = (16, 16, 1)
GRID_3D  = (math.ceil(IMG_W / 16), math.ceil(IMG_H / 16), math.ceil(BATCH / 1))
print(f'Grid 3D: {GRID_3D[0]}x{GRID_3D[1]}x{GRID_3D[2]} bloques')

mean = float(np.mean(h_tensor))
std  = float(np.std(h_tensor))

kernel_batch_normalize_3d[GRID_3D, BLOCK_3D](d_tensor, mean, std, d_out_bn, BATCH, IMG_H, IMG_W)
cuda.synchronize()

kernel_relu_3d[GRID_3D, BLOCK_3D](d_out_bn, d_out_relu, BATCH, IMG_H, IMG_W)
cuda.synchronize()

h_bn   = d_out_bn.copy_to_host()
h_relu = d_out_relu.copy_to_host()

expected_bn = (h_tensor - mean) / (std + 1e-8)
print(f'Batch Norm correcto : {np.allclose(h_bn, expected_bn, atol=1e-5)}')
print(f'ReLU min value      : {h_relu.min():.4f} (debe ser >= 0)')
print(f'Valores > 0: {(h_relu > 0).sum():,} / {h_relu.size:,}')

In [ ]:
# ── EJEMPLO 3B: Tensores 4D con CuPy — formato CNN ─────────────────
# Formato (N, C, H, W): batch × canales × alto × ancho

N, C, H2, W2 = 64, 256, 28, 28
d_feat = cp.random.randn(N, C, H2, W2, dtype=cp.float32)  # directo en GPU
print(f'Tensor 4D: {d_feat.shape}')
print(f'Memoria GPU: {d_feat.nbytes / 1024**2:.1f} MB')

# Normalización espacial: mean/std sobre H y W por (N, C)
mean_2d = d_feat.mean(axis=(2, 3), keepdims=True)  # (N, C, 1, 1)
std_2d  = d_feat.std(axis=(2, 3), keepdims=True)
d_norm  = (d_feat - mean_2d) / (std_2d + 1e-8)

# Global Average Pooling: (N, C, H, W) -> (N, C)
d_gap = d_feat.mean(axis=(2, 3))

# Softmax sobre canales
d_exp      = cp.exp(d_gap - d_gap.max(axis=1, keepdims=True))
d_softmax  = d_exp / d_exp.sum(axis=1, keepdims=True)

cp.cuda.Stream.null.synchronize()

print(f'Tensor normalizado : {d_norm.shape}')
print(f'Global Avg Pool    : {d_gap.shape}')
print(f'Softmax            : {d_softmax.shape}')
print(f'Softmax suma[0]    : {float(d_softmax[0].sum()):.6f}  (debe ser 1.0)')

predictions = cp.asnumpy(d_softmax)
top_class   = predictions.argmax(axis=1)
print(f'Clase predicha (ejemplo 0): {top_class[0]} de {C} canales')

---
## Sección 4 — Benchmarking Sistemático CPU vs GPU

In [ ]:
# Benchmark completo: kernel puro vs total (kernel + transferencias)

def benchmark_op(op_name, cpu_fn, gpu_fn, sizes):
    results = []
    for size in sizes:
        data_cpu = np.random.randn(size).astype(np.float32)

        t0 = time.perf_counter()
        _ = cpu_fn(data_cpu)
        t_cpu = (time.perf_counter() - t0) * 1000

        # GPU: solo kernel (datos ya en VRAM)
        data_gpu = cp.asarray(data_cpu)
        _ = gpu_fn(data_gpu); cp.cuda.Stream.null.synchronize()  # warmup
        t0 = time.perf_counter()
        _ = gpu_fn(data_gpu)
        cp.cuda.Stream.null.synchronize()
        t_kernel = (time.perf_counter() - t0) * 1000

        # GPU: total incluyendo H2D + kernel + D2H
        t0 = time.perf_counter()
        data_gpu2 = cp.asarray(data_cpu)
        res = gpu_fn(data_gpu2)
        _ = cp.asnumpy(res)
        cp.cuda.Stream.null.synchronize()
        t_total = (time.perf_counter() - t0) * 1000

        results.append({
            'op': op_name, 'N': size,
            'CPU (ms)': round(t_cpu, 3),
            'GPU kernel (ms)': round(t_kernel, 3),
            'GPU total (ms)': round(t_total, 3),
            'Speedup kernel': round(t_cpu / t_kernel, 1),
            'Speedup total':  round(t_cpu / t_total, 1),
        })
    return results


sizes = [1_000, 10_000, 100_000, 1_000_000, 10_000_000]
all_results = []
all_results += benchmark_op('Suma',         lambda x: x + x,       lambda x: x + x,       sizes)
all_results += benchmark_op('Exp',          lambda x: np.exp(x),    lambda x: cp.exp(x),   sizes)
all_results += benchmark_op('Sum reduce',   lambda x: np.sum(x),    lambda x: cp.sum(x),   sizes)
all_results += benchmark_op('Sort',         lambda x: np.sort(x),   lambda x: cp.sort(x),  sizes)

# Imprimir tabla por operación
import pandas as pd
df = pd.DataFrame(all_results)
for op in df['op'].unique():
    print(f'--- {op} ---')
    sub = df[df['op'] == op][['N', 'CPU (ms)', 'GPU kernel (ms)', 'GPU total (ms)', 'Speedup kernel', 'Speedup total']]
    print(sub.to_string(index=False))
    print()

In [ ]:
# Gráfico: speedup vs tamaño para cada operación

ops = df['op'].unique()
fig, axes = plt.subplots(1, len(ops), figsize=(16, 4), sharey=False)

for ax, op in zip(axes, ops):
    sub = df[df['op'] == op]
    ax.semilogx(sub['N'], sub['Speedup kernel'], 'o-', color='#00D4FF', lw=2, label='Solo kernel')
    ax.semilogx(sub['N'], sub['Speedup total'],  's--', color='#F59E0B', lw=2, label='Kernel+Transfer')
    ax.axhline(1, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(op); ax.set_xlabel('N'); ax.set_ylabel('Speedup (x)')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('CPU vs GPU — Speedup por operacion y tamano', y=1.02)
plt.tight_layout()
plt.show()

print()
print('Observacion clave: el speedup del kernel puro >> speedup total.')
print('El overhead de transferencia PCIe domina para N pequeños.')

---
## Sección 5 — GPU en Machine Learning con PyTorch

PyTorch abstrae los kernels CUDA pero los conceptos son los mismos:
`tensor.to('cuda')` = H2D, `tensor.cpu()` = D2H, y cada operación lanza kernels internos.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando device: {device}')

# Crear tensores directamente en GPU (sin pasar por CPU)
x = torch.randn(1000, 1000, device=device)
y = torch.randn(1000, 1000, device=device)
z = torch.mm(x, y)  # GEMM via cuBLAS
print(f'Matmul 1000x1000 en GPU: {z.shape}, device: {z.device}')

# H2D y D2H explícitos
cpu_tensor = torch.randn(10_000)
gpu_tensor = cpu_tensor.to(device)   # H2D
cpu_back   = gpu_tensor.cpu()        # D2H
print(f'Valores preservados: {torch.allclose(cpu_tensor, cpu_back)}')

In [ ]:
# ── Entrenamiento de red neuronal en CPU vs GPU ────────────────────

class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim)
        )
    def forward(self, x):
        return self.net(x)


def train_benchmark(device_name, epochs=20, batch=2048, n=50000):
    dev   = torch.device(device_name)
    model = MLP(128, 512, 10).to(dev)
    opt   = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    X = torch.randn(n, 128, device=dev)
    y = torch.randn(n, 10,  device=dev)

    t0 = time.perf_counter()
    for _ in range(epochs):
        for i in range(0, n, batch):
            xb, yb = X[i:i+batch], y[i:i+batch]
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    if device_name == 'cuda': torch.cuda.synchronize()
    return time.perf_counter() - t0, float(loss.item())


print('Entrenando en CPU...')
t_cpu, l_cpu = train_benchmark('cpu')
print(f'  CPU: {t_cpu:.2f}s  loss: {l_cpu:.4f}')

print('Entrenando en GPU...')
t_gpu, l_gpu = train_benchmark('cuda')
print(f'  GPU: {t_gpu:.2f}s  loss: {l_gpu:.4f}')

print(f'Speedup entrenamiento: {t_cpu/t_gpu:.1f}x')

In [ ]:
# ── Monitor de memoria GPU ────────────────────────────────────────

def gpu_mem(label=''):
    a = torch.cuda.memory_allocated()  / 1024**2
    c = torch.cuda.memory_reserved()   / 1024**2
    t = torch.cuda.get_device_properties(0).total_memory / 1024**2
    print(f'[{label:22s}]  Asignado: {a:6.1f} MB  Caché: {c:6.1f} MB  Total: {t:.0f} MB')

torch.cuda.empty_cache()
gpu_mem('Inicio')

t1 = torch.randn(1000, 1000, device='cuda')
gpu_mem('+ 1M floats')

t2 = torch.randn(4096, 4096, device='cuda')
gpu_mem('+ 16M floats')

model = nn.Sequential(nn.Linear(4096, 4096), nn.ReLU(), nn.Linear(4096, 1000)).cuda()
gpu_mem('+ Modelo grande')

del t1, t2, model
torch.cuda.empty_cache()
gpu_mem('Tras liberar')

---
## Sección 6 — Kernel CUDA C embebido en Python (CuPy RawKernel)

CuPy permite escribir kernels CUDA en C directamente. La sintaxis es idéntica a la teoría.

In [ ]:
# Kernel 1D: función sigmoide
sigmoid_kernel = cp.RawKernel(r'''
extern "C" __global__
void sigmoid(const float* x, float* out, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        out[i] = 1.0f / (1.0f + expf(-x[i]));
    }
}
''', 'sigmoid')

# Kernel 2D: suma de matrices
matsum_kernel = cp.RawKernel(r'''
extern "C" __global__
void matsum(const float* A, const float* B, float* C, int rows, int cols) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (col < cols && row < rows) {
        int idx = row * cols + col;
        C[idx] = A[idx] + B[idx];
    }
}
''', 'matsum')

# ── Usar kernel sigmoide ──────────────────────────────────────────
N       = 1_000_000
x_gpu   = cp.random.uniform(-5, 5, N, dtype=cp.float32)
out_gpu = cp.empty(N, dtype=cp.float32)

BLOCK = 256
GRID  = (N + BLOCK - 1) // BLOCK

sigmoid_kernel((GRID,), (BLOCK,), (x_gpu, out_gpu, N))
cp.cuda.Stream.null.synchronize()

from scipy.special import expit
expected = expit(cp.asnumpy(x_gpu)).astype(np.float32)
print('Kernel Sigmoid (CUDA C en Python):')
print(f'  Correcto: {np.allclose(cp.asnumpy(out_gpu), expected, atol=1e-5)}')
print(f'  Rango: [{float(out_gpu.min()):.4f}, {float(out_gpu.max()):.4f}]  (debe ser [0, 1])')

# ── Usar kernel 2D ────────────────────────────────────────────────
ROWS, COLS = 512, 512
A_gpu = cp.random.randn(ROWS, COLS, dtype=cp.float32)
B_gpu = cp.random.randn(ROWS, COLS, dtype=cp.float32)
C_gpu = cp.empty((ROWS, COLS), dtype=cp.float32)

BLOCK2D = (16, 16)
GRID2D  = (math.ceil(COLS / 16), math.ceil(ROWS / 16))

matsum_kernel(GRID2D, BLOCK2D, (A_gpu, B_gpu, C_gpu, ROWS, COLS))
cp.cuda.Stream.null.synchronize()

print(f'Kernel MatSum 2D {ROWS}x{COLS}: correcto = {cp.allclose(C_gpu, A_gpu + B_gpu)}')

---
## Sección 7 — Resumen visual y tabla de correspondencias

In [ ]:
# Gráfico resumen: conceptos clave

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0A0E1A')

# ── 1. CPU vs GPU: tiempo vs N ───────────────────────────────────
ax = axes[0]
ax.set_facecolor('#111827')
ns    = np.logspace(3, 7, 20).astype(int)
cpu_t = ns / 5e7
gpu_t = ns / 2e9 + 0.0005
ax.loglog(ns, cpu_t * 1000, color='#F59E0B', lw=2.5, label='CPU')
ax.loglog(ns, gpu_t * 1000, color='#00D4FF', lw=2.5, label='GPU (kernel+transfer)')
ax.axvline(x=5e4, color='#10B981', linestyle='--', alpha=0.7, label='Breakeven ~50K')
ax.set_xlabel('N elementos', color='white')
ax.set_ylabel('Tiempo (ms)', color='white')
ax.set_title('Cuándo vale la GPU', color='white', pad=10)
ax.legend(facecolor='#1A2235', labelcolor='white')
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_color('#1E3A5F')
ax.grid(alpha=0.2, color='#1E3A5F')

# ── 2. Jerarquía de memoria ───────────────────────────────────────
ax = axes[1]
ax.set_facecolor('#111827')
mem_types  = ['Registros', 'Shared', 'L1/L2', 'Global VRAM', 'RAM CPU']
bandwidths = [8000, 3500, 2000, 900, 50]
colors_bar = ['#10B981', '#00D4FF', '#7C3AED', '#F59E0B', '#94A3B8']
bars = ax.barh(mem_types, bandwidths, color=colors_bar, alpha=0.85, edgecolor='none')
for bar, val in zip(bars, bandwidths):
    ax.text(val + 60, bar.get_y() + bar.get_height()/2,
            f'{val} GB/s', va='center', color='white', fontsize=9)
ax.set_xlabel('Ancho de banda (GB/s)', color='white')
ax.set_title('Jerarquía de memoria GPU', color='white', pad=10)
ax.tick_params(colors='white'); ax.set_xlim(0, 10500)
for spine in ax.spines.values(): spine.set_color('#1E3A5F')
ax.grid(alpha=0.2, color='#1E3A5F', axis='x')

plt.tight_layout()
plt.show()

In [ ]:
# Tabla de correspondencias CPU <-> GPU

summary = '''
+------------------------------------------------------------------+
|           MAPA DE CONCEPTOS: CPU  <->  GPU (Python)              |
+--------------------+--------------------+------------------------+
|  CPU (NumPy/Python)|  GPU Numba         |  GPU CuPy/PyTorch      |
+--------------------+--------------------+------------------------+
|  for i in range(N) |  @cuda.jit kernel  |  cp.func(arr)          |
|  for row, for col  |  kernel 2D (bIdx)  |  cp.func (broadcasting)|
|  arr = np.zeros(N) |  cuda.device_array |  cp.zeros(N)           |
|  arr (en RAM)      |  cuda.to_device()  |  cp.asarray() / .cuda()|
|  result (en RAM)   |  .copy_to_host()   |  cp.asnumpy() / .cpu() |
|  func(data)        |  kernel[G,B](data) |  cp.func(data)         |
|  float arr[256]    |  cuda.shared.array |  (gestionado interno)   |
+--------------------+--------------------+------------------------+

REGLAS PRACTICAS:
  - GPU vale la pena cuando N > ~50,000 elementos
  - Minimizar transferencias H2D/D2H: operar en GPU lo mas posible
  - blockDim tipico: 128, 256 o 512 hilos (multiplo de 32)
  - Usar memoria compartida cuando el bloque reutiliza datos
  - Guardia siempre: if i < N: ...
'''
print(summary)